# Cancer Incidence Trends in Pune, India (1993–2017)

## Research Question
**How has the burden of cancer incidences in the city of Pune changed over a 25-year period from 1993–2017, and how do these patterns differ by sex, age, and cancer type?**

## About Pune

Pune is the second-largest city in the state of Maharashtra and one of India's most populous urban centers. It is widely known as an educational hub — sometimes called the "Oxford of the East" — home to numerous universities and research institutes, alongside a thriving information technology and manufacturing industry sector.

Pune's population has grown rapidly over the past few decades, driven largely by migration of students and professionals. Pune has a relatively young demographic profile with a median age in the late 20s. This combination of rapid urbanization, a large population base, and long-standing institutional infrastructure makes Pune's population-based cancer registry one of the longest-running in India which provides a particularly valuable source for tracking long-term cancer trends.

## About IACR Cancer Registries and the CI5 Series

The International Association of Cancer Registries (IACR), in collaboration with the International Agency for Research on Cancer (IARC), maintains the *Cancer Incidence in Five Continents* (CI5) series — a set of volumes published roughly every five years that compiles cancer data from population-based registries around the world.

A population-based cancer registry systematically records cancer incidences within a defined geographic area providing a broader picture of how common each cancer type is in a community.

The Pune (Poona) registry has contributed data to the CI5 series since Volume VIII (1993), making it possible to trace over two decades of cancer trends in this specific urban population.

This kind of data is valuable because it informs public health planning by helping researchers and health officials answer practical questions such as which cancers are becoming more or less common, which age groups or sexes are most affected, and where prevention or screening resources should be directed.

## Objective
This notebook extracts, cleans, and analyzes population-based cancer registry data for Pune from five successive volumes of the IARC's *Cancer Incidence in Five Continents* (CI5) series (Volumes VIII through XII).

The goal is to:

- Build a single, consistent dataset from five differently formatted source files, standardizing cancer codes, age groups, and registration periods across volumes.
- Calculate crude cancer incidence rates (cases per 100,000 people) overall, by sex, and by age group, to track how cancer burden evolved across the five registration periods (1993-1997 through 2013-2017).
- Identify the 10 most frequently diagnosed cancer types in Pune over this period and examine how each one trended over time and across age groups.

## Data Source
All data originates from the International Agency for Research on Cancer's (IARC) *Cancer Incidence in Five Continents* database, covering the Pune (Poona) population-based cancer registry. Source files: https://ci5.iarc.fr

## Loading Required Tools and Libraries

This cell loads all the external tools (Python libraries) that the rest of the script depends on.

- `os`, `zipfile`, `re`, and `csv` are general-purpose tools used later for navigating folders, opening zip files, matching text patterns, and reading CSV data.
- `numpy` and `pandas` are the core data-handling libraries — `pandas` organizes data into tables (like spreadsheets), while `numpy` handles numerical calculations behind the scenes.
- `matplotlib` (along with its `pyplot`, `ticker`, `patches`, `colors`, `cm`, and `lines` components) is the charting library used throughout the script to build all the bar charts, line charts, and stacked plots.
- `warnings.filterwarnings('ignore')` silences non-critical warning messages to avoid clutter the output.
- `%matplotlib inline` tells the notebook to display charts directly below each code cell rather than in a separate window.

In [ ]:
import os
import zipfile
import re
import warnings
import csv
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
from matplotlib.colors import Normalize
warnings.filterwarnings('ignore')
%matplotlib inline
import matplotlib.cm as cm
import matplotlib.lines as mlines

## Setting Up File Paths and Names

This cell tells the script where to find things and what to call the files it creates.

- It locates the folder where the notebook itself lives, and uses that as the "home base" for all file operations.
- It builds a list of the five zipped data files (`CI5-VIIId.zip` through `CI5-XII.zip`), each one containing cancer registry records for a specific time period covered by the CI5 volumes.
- It stores the special identification codes used to pick out "Pune" records within each registry file — different volumes use different code formats.
- It records today's date and uses it to automatically stamp every output file name.
- It defines the exact output file names the script will eventually produce — this includes per-volume extracted data tables, combined summary tables (crude rates, age groups, top cancer sites), and image files for charts.
- Finally, it checks whether each of the five zip files actually exists in the folder and prints a quick confirmation ("found" or "NOT FOUND") for each one.

In [ ]:
# Base Directory
BASE_DIR = os.path.dirname(os.path.abspath("sample.ipynb"))

# Zip files for extracting Pune data
ZIPS = {
    "VIII": os.path.join(BASE_DIR, "CI5-VIIId.zip"),
    "IX":   os.path.join(BASE_DIR, "CI5-IXd.zip"),
    "X":    os.path.join(BASE_DIR, "CI5-Xd.zip"),
    "XI":   os.path.join(BASE_DIR, "CI5-XI.zip"),
    "XII":  os.path.join(BASE_DIR, "CI5-XII.zip"),
}

# Codes for Pune in different registries
PUNE_CODE_VIII     = 91
PUNE_FILE_IX_X     = "43560699"
PUNE_CODE_XI_XII   = 435600699

# Output file names - date tag updates automatically on each run, name stays short and constant
RUN_DATE = pd.Timestamp.today().strftime("%Y-%m-%d")

def out_name(short_name: str) -> str:
    return os.path.join(BASE_DIR, f"{RUN_DATE}_{short_name}")

# Per-volume extracted data
OUT_IACR_VIII = out_name("iarc_pune_VIII_1993-1997.csv")
OUT_IACR_IX   = out_name("iarc_pune_IX_1998-2002.csv")
OUT_IACR_X    = out_name("iarc_pune_X_2003-2007.csv")
OUT_IACR_XI   = out_name("iarc_pune_XI_2008-2011.csv")
OUT_IACR_XII  = out_name("iarc_pune_XII_2013-2017.csv")

# Compiled all-sites tables
OUT_LABELS             = out_name("cancer_labels_by_volumes.csv")
OUT_CRUDE_RATE         = out_name("pune_crude_rate.csv")
OUT_AGE                = out_name("pune_all_sites.csv")
OUT_AGEBAND15          = out_name("pune_by_age_groups.csv")

# Label filtering / common-sites / top10 outputs
OUT_COMMON              = out_name("cancer_labels.csv")
OUT_TISSUE_MAIN         = out_name("pune_common_sites.csv")
OUT_TOP10               = out_name("pune_top10_sites.csv")
OUT_AGE_DIST            = out_name("pune_top10_sites_age_distribution.csv")

# Figures
OUT_FIG_POP             = out_name("population_trends.png")
OUT_FIG_SEX             = out_name("crude_rate_by_sex.png")
OUT_FIG_AGEBAND         = out_name("age_band_stacked_bar.png")
OUT_FIG_AGEBAND_RATE    = out_name("age_band_crude_rate.png")
OUT_TOP10_FIG_FEMALE    = out_name("top10_sites_female_trend.png")
OUT_TOP10_FIG_MALE      = out_name("top10_sites_male_trend.png")
OUT_AGE_FIG_FEMALE      = out_name("top10_age_dist_female.png")
OUT_AGE_FIG_MALE        = out_name("top10_age_dist_male.png")

# Verify ZIP files exist in the same folder as the script
for vol, path in ZIPS.items():
    status = "found" if os.path.exists(path) else "NOT FOUND"
    print(f"  CI5-{vol}  {status}  ({path})")

## Reading Cancer Type Labels from Each Volume

This cell builds a "dictionary" for each CI5 volume that translates numeric cancer codes into human-readable cancer type names (like "Lung" or "All sites").

- Each registry volume stores its list of cancer types in a slightly different text format. Three separate parsing functions handle these differences: one for Volume VIII, one shared by Volumes IX and X, and one shared by Volumes XI and XII.
- `parse_cancer_viii` reads lines formatted like "1 C00-97 All sites," pulling out the numeric code and the plain-language label while ignoring the ICD classification code in between.
- `parse_cancer_ix_x` reads lines formatted like "001 All sites (C00-96)," extracting the code and label while stripping out the ICD code shown in parentheses at the end.
- `parse_cancer_xi_xii` reads tab-separated lines where the code, ICD classification, and label sit in separate columns, and pulls out just the code and label.
- The script then opens each zip file, finds the specific text file inside it that contains the cancer type listings, and runs the matching parser on it to build a lookup table (code → cancer name) for that volume.
- Finally, it prints how many cancer labels were found in each volume and shows what code 1 translates to, as a quick sanity check that the parsing worked correctly.

In [ ]:
def parse_cancer_viii(raw_text):
    """Vol. VIII: '  1 C00-97  All sites' — integer code + ICD code + label"""
    d = {}
    for line in raw_text.split("\n"):
        line = line.strip()
        if not line or line.lower().startswith("cancer"):
            continue
        m = re.match(r"^(\d+)\s+\S+\s+(.*)", line)
        if m:
            d[int(m.group(1))] = m.group(2).strip()
    return d


def parse_cancer_ix_x(raw_text):
    """Vol. IX/X: '001 All sites (C00-96)' — zero-padded code + label (ICD in parens)"""
    d = {}
    for line in raw_text.split("\n"):
        line = line.strip()
        if not line:
            continue
        m = re.match(r"^(\d+)\s+(.*)", line)
        if m:
            code  = int(m.group(1))
            label = re.sub(r"\s*\(C[\d\-,\s]+\)\s*$", "", m.group(2)).strip()
            d[code] = label
    return d


def parse_cancer_xi_xii(raw_text):
    """Vol. XI/XII: tab-separated — col0=code, col1=ICD, col2=label"""
    d = {}
    for line in raw_text.split("\n"):
        parts = line.strip().split("\t")
        if len(parts) >= 3 and parts[0].strip().isdigit():
            d[int(parts[0].strip())] = parts[2].strip()
    return d


# ── Load all dictionaries ─────────────────────────────────────────────────
CANCER_DICTS = {}
PARSERS = {
    "VIII": parse_cancer_viii,
    "IX":   parse_cancer_ix_x,
    "X":    parse_cancer_ix_x,
    "XI":   parse_cancer_xi_xii,
    "XII":  parse_cancer_xi_xii,
}

for vol, parser in PARSERS.items():
    with zipfile.ZipFile(ZIPS[vol]) as z:
        cancer_file = next(n for n in z.namelist() if "cancer" in n.lower())
        with z.open(cancer_file) as f:
            raw = f.read().decode("latin-1", errors="ignore")
    CANCER_DICTS[vol] = parser(raw)
    print(f"CI5-{vol}: {len(CANCER_DICTS[vol])} labels parsed  ",
          f"(code 1 = '{CANCER_DICTS[vol].get(1)}')")

## Setting Up Shared Reference Tables

This cell defines a set of common lookup values and a helper function that will be reused throughout the rest of the script to keep the data consistent and organized.

- `AGE_COLS` lists the 18 age brackets used in the raw registry data, in the exact order they appear in the original files (e.g. "0_4," "5_9," up through "85" for 85 and older).
- `AGE_LABELS` converts those raw age codes into cleaner, easier-to-read labels for reports and charts, like turning "0_4" into "0-4" and "85" into "85+".
- `AGE_MAP` and `SEX_MAP` act as translation keys: they convert the numeric codes used in the raw files (like age group "1" or sex "2") into their real-world meanings (the actual age bracket, or "Male"/"Female").
- `PERIODS` and `MIDPOINTS` connect each registry volume (VIII through XII) to the actual years it covers (e.g. Volume X = 2003-2007) and a single "midpoint year" that represents that period on a timeline, which is useful for plotting trends later.
- The `make_row` function is a reusable building block: given basic facts about a single record (which volume, sex, age group, cancer type, number of cases, and population size), it calculates the cancer rate per 100,000 people and packages everything into one neat, labeled row ready to be added to a table.

In [ ]:
# Shared constants used by all extractor functions
AGE_COLS = [
    "0_4",  "5_9",  "10_14", "15_19", "20_24", "25_29", "30_34",
    "35_39", "40_44", "45_49", "50_54", "55_59", "60_64",
    "65_69", "70_74", "75_79", "80_84", "85",
]

AGE_LABELS = {
    "0_4":"0-4",    "5_9":"5-9",    "10_14":"10-14", "15_19":"15-19",
    "20_24":"20-24", "25_29":"25-29", "30_34":"30-34", "35_39":"35-39",
    "40_44":"40-44", "45_49":"45-49", "50_54":"50-54", "55_59":"55-59",
    "60_64":"60-64", "65_69":"65-69", "70_74":"70-74", "75_79":"75-79",
    "80_84":"80-84", "85":"85+",
}

AGE_MAP = {i + 1: c for i, c in enumerate(AGE_COLS)}  # {1:'0_4', ..., 18:'85'}
SEX_MAP = {1: "Male", 2: "Female"}

PERIODS   = {
    "VIII": "1993-1997", "IX": "1998-2002", "X": "2003-2007",
    "XI":   "2008-2011", "XII": "2013-2017",
}
MIDPOINTS = {
    "1993-1997": 1995, "1998-2002": 2000, "2003-2007": 2005,
    "2008-2011": 2010, "2013-2017": 2015,
}


def make_row(vol, sex_label, age_key, cancer_code, cancer_label, cases, population):
    period = PERIODS[vol]
    rate   = round(cases / population * 100_000, 4) if population > 0 else None
    return {
        "Volume":        vol,
        "Period":        period,
        "Year_Midpoint": MIDPOINTS[period],
        "Registry":      "Pune (Poona)",
        "Sex":           sex_label,
        "Age_Group":     AGE_LABELS[age_key],
        "Age_Group_Sort":AGE_COLS.index(age_key) + 1,
        "Cancer_Code":   cancer_code,
        "Cancer_Label":  cancer_label,
        "Cases":         int(cases),
        "Population":    int(population),
        "Rate_per_100k": rate,
    }

print("Shared constants ready")

## Extracting Pune-Specific Data from Each Volume

This is the core extraction step: it digs into all five zipped registry files, pulls out only the records belonging to Pune, and reshapes them into one consistent, easy-to-read table per volume.

- **Volume VIII (1993-1997):** Opens the zip file and reads its single combined data file, then filters it down to just the rows where the registry code matches Pune. For each combination of sex and cancer type, it looks up the plain-language cancer name and builds a clean row (with age group, case count, and population) using the `make_row` helper, then saves the result as its own CSV.
- **Volumes IX and X (1998-2002 and 2003-2007):** Follows the same idea, but these volumes store Pune's data in its own separate file inside the zip (identified by a special file code).  The script finds that specific file first before reading and reshaping it the same way.
- **Volumes XI and XII (2008-2011 and 2013-2017):** These newer volumes split information into two separate files — one for cancer case counts and one for population counts. The script loads both, filters each to Pune's registry code, and builds a lookup table matching population figures to each sex and age group. It then combines case counts with the matching population numbers for every cancer type and age group to build the same standardized row format as the earlier volumes.
- After processing each volume, the resulting table (sex, age group, cancer type, cases, population, and calculated rate) is saved to its own CSV file, and a confirmation message prints once each file is written.
- By the end of this cell, there are five separate, cleanly formatted CSV files — one per volume — all sharing the exact same structure, which makes it possible to combine or compare them easily in later steps.

In [ ]:
# Data extraction
# CI5-VIII (1993–1997)
with zipfile.ZipFile(ZIPS["VIII"]) as z:
    with z.open("CI5-VIII.csv") as f:
        raw_viii = pd.read_csv(
            f, header=None,
            names=["registry","sex","cancer","age_group","cases","person_years"]
        )

rows = []
pune_viii = raw_viii[raw_viii["registry"] == PUNE_CODE_VIII]
for sex_code, sex_label in SEX_MAP.items():
    s = pune_viii[pune_viii["sex"] == sex_code]
    for cancer_code in sorted(s["cancer"].unique()):
        label = CANCER_DICTS["VIII"].get(int(cancer_code), f"Code_{cancer_code}")
        for _, r in s[s["cancer"] == cancer_code].iterrows():
            ag = int(r["age_group"])
            if ag not in AGE_MAP:
                continue
            rows.append(make_row("VIII", sex_label, AGE_MAP[ag], int(cancer_code), label, r["cases"], r["person_years"]))

df_viii = pd.DataFrame(rows)
df_viii.to_csv(OUT_IACR_VIII, index=False)
print(f"Saved {OUT_IACR_VIII}")

# CI5-IX (1998–2002)
with zipfile.ZipFile(ZIPS["IX"]) as z:
    pune_file_ix = next(n for n in z.namelist() if PUNE_FILE_IX_X in n)
    with z.open(pune_file_ix) as f:
        raw_ix = pd.read_csv(
            f, header=None,
            names=["sex","cancer","age_group","cases","person_years"]
        )

rows = []
for sex_code, sex_label in SEX_MAP.items():
    s = raw_ix[raw_ix["sex"] == sex_code]
    for cancer_code in sorted(s["cancer"].unique()):
        label = CANCER_DICTS["IX"].get(int(cancer_code), f"Code_{cancer_code}")
        for _, r in s[s["cancer"] == cancer_code].iterrows():
            ag = int(r["age_group"])
            if ag not in AGE_MAP:
                continue
            rows.append(make_row("IX", sex_label, AGE_MAP[ag], int(cancer_code), label, r["cases"], r["person_years"]))

df_ix = pd.DataFrame(rows)
df_ix.to_csv(OUT_IACR_IX, index=False)
print(f"Saved {OUT_IACR_IX}")

# CI5-X (2003–2007)
with zipfile.ZipFile(ZIPS["X"]) as z:
    pune_file_x = next(n for n in z.namelist() if PUNE_FILE_IX_X in n)
    with z.open(pune_file_x) as f:
        raw_x = pd.read_csv(
            f, header=None,
            names=["sex","cancer","age_group","cases","person_years"]
        )

rows = []
for sex_code, sex_label in SEX_MAP.items():
    s = raw_x[raw_x["sex"] == sex_code]
    for cancer_code in sorted(s["cancer"].unique()):
        label = CANCER_DICTS["X"].get(int(cancer_code), f"Code_{cancer_code}")
        for _, r in s[s["cancer"] == cancer_code].iterrows():
            ag = int(r["age_group"])
            if ag not in AGE_MAP:
                continue
            rows.append(make_row("X", sex_label, AGE_MAP[ag], int(cancer_code), label, r["cases"], r["person_years"]))

df_x = pd.DataFrame(rows)
df_x.to_csv(OUT_IACR_X, index=False)
print(f"Saved {OUT_IACR_X}")

# CI5-XI (2008–2011)
with zipfile.ZipFile(ZIPS["XI"]) as z:
    with z.open("cases.csv") as f:
        cases_xi = pd.read_csv(f)
    with z.open("pop.csv") as f:
        pop_xi = pd.read_csv(f)

pune_cases_xi = cases_xi[cases_xi["REGISTRY"] == PUNE_CODE_XI_XII]
pune_pop_xi   = pop_xi[pop_xi["REGISTRY"]   == PUNE_CODE_XI_XII]

pop_lkp_xi = {
    int(r["SEX"]): {age_key: int(r.get(f"P{age_key}", 0)) for age_key in AGE_COLS}
    for _, r in pune_pop_xi.iterrows()
}

rows = []
for sex_code, sex_label in SEX_MAP.items():
    s = pune_cases_xi[pune_cases_xi["SEX"] == sex_code]
    p = pop_lkp_xi.get(sex_code, {})
    for _, crow in s.iterrows():
        cancer_code = int(crow["CANCER"])
        label = CANCER_DICTS["XI"].get(cancer_code, f"Code_{cancer_code}").strip()
        for age_key in AGE_COLS:
            rows.append(make_row("XI", sex_label, age_key, cancer_code, label,
                                 int(crow.get(f"N{age_key}", 0)), p.get(age_key, 0)))

df_xi = pd.DataFrame(rows)
df_xi.to_csv(OUT_IACR_XI, index=False)
print(f"Saved {OUT_IACR_XI}")

# CI5-XII (2013–2017)
with zipfile.ZipFile(ZIPS["XII"]) as z:
    with z.open("cases.csv") as f:
        cases_xii = pd.read_csv(f)
    with z.open("pop.csv") as f:
        pop_xii = pd.read_csv(f)

pune_cases_xii = cases_xii[cases_xii["REGISTRY"] == PUNE_CODE_XI_XII]
pune_pop_xii   = pop_xii[pop_xii["REGISTRY"]   == PUNE_CODE_XI_XII]

pop_lkp_xii = {
    int(r["SEX"]): {age_key: int(r.get(f"P{age_key}", 0)) for age_key in AGE_COLS}
    for _, r in pune_pop_xii.iterrows()
}

rows = []
for sex_code, sex_label in SEX_MAP.items():
    s = pune_cases_xii[pune_cases_xii["SEX"] == sex_code]
    p = pop_lkp_xii.get(sex_code, {})
    for _, crow in s.iterrows():
        cancer_code = int(crow["CANCER"])
        label = CANCER_DICTS["XII"].get(cancer_code, f"Code_{cancer_code}").strip()
        for age_key in AGE_COLS:
            rows.append(make_row("XII", sex_label, age_key, cancer_code, label,
                                 int(crow.get(f"N{age_key}", 0)), p.get(age_key, 0)))

df_xii = pd.DataFrame(rows)
df_xii.to_csv(OUT_IACR_XII, index=False)
print(f"Saved {OUT_IACR_XII}")

## Calculating Overall Cancer Rates Over Time

This cell combines all five volumes into one dataset and calculates the overall ("all sites") cancer rate for each time period by sex.

- It loads the five per-volume CSV files created in the previous step and stacks them together into one large combined table covering all years from 1993 to 2017.
- It narrows this table down to only the rows labeled "All sites," which represents the total cancer burden rather than any single specific cancer type.
- It groups the data by time period and sex, then adds up the total number of cases and the total population within each group.
- Using those totals, it calculates the crude cancer rate — the number of cases per 100,000 people — for each period and sex combination, which is a standard way to compare cancer burden across groups of different population sizes.
- It puts the time periods in proper chronological order (rather than alphabetical) and sorts the table accordingly, then saves the final result as a CSV file and prints it out for a quick visual check.

In [ ]:
# Compute all-sites crude rate per 100k by period × sex
vol_files = {
    "VIII": OUT_IACR_VIII, "IX": OUT_IACR_IX, "X": OUT_IACR_X,
    "XI": OUT_IACR_XI, "XII": OUT_IACR_XII,
}

df_all = pd.concat(
    [pd.read_csv(f) for f in vol_files.values()],
    ignore_index=True,
)

# Filter to All sites only and excluding 80-84 and 85+ age groups
df_allsites = df_all[
    (df_all["Cancer_Label"] == "All sites") &
    (~df_all["Age_Group"].isin(["80-84", "85+"]))
].copy()

# Aggregate: sum Cases and Population per Period × Sex
agg = (
    df_allsites
    .groupby(["Period", "Year_Midpoint", "Volume", "Sex"], sort=False)
    .agg(Total_Cases=("Cases", "sum"), Total_Population=("Population", "sum"))
    .reset_index()
)

# Compute crude rate: Total_Cases / Total_Population × 100,000
agg["Crude_Rate_per_100k"] = (
    agg["Total_Cases"] / agg["Total_Population"] * 100000
).round(4)

# Sort by period order
period_order = ["1993-1997", "1998-2002", "2003-2007", "2008-2011", "2013-2017"]
agg["Period"] = pd.Categorical(agg["Period"], categories=period_order, ordered=True)
agg = agg.sort_values(["Period", "Sex"]).reset_index(drop=True)

# Save
agg.to_csv(OUT_CRUDE_RATE, index=False)
print("All-Sites Crude Rate per 100,000 — Pune Registry\n")
print(agg.to_string(index=False))
print(f"\n Saved")

## Breaking Down Cancer Rates by Age Group

This cell zooms into finer detail, calculating the "all sites" cancer rate separately for each of the 18 age brackets, within every time period and sex.

- It groups the "all sites" data by time period, sex, and age group simultaneously, then sums up the total cases and total population within each of these narrower groups.
- It calculates the crude cancer rate (cases per 100,000 people) for each period-sex-age combination, the same way as before, but now at a much more detailed level.
- If any group has zero recorded population (which would make the rate calculation meaningless), the rate is left blank instead of showing a misleading number.
- It sorts everything in a logical order — first by time period chronologically, then by sex, then by age group from youngest to oldest.
- The detailed table is saved as its own CSV file, and the cell prints the full table along with a row count.

In [ ]:
# Aggregate by Period x Sex x Age_Group
agg_age = (
    df_allsites
    .groupby(["Period", "Year_Midpoint", "Volume", "Sex", "Age_Group", "Age_Group_Sort"],
             sort=False)
    .agg(Total_Cases=("Cases", "sum"), Total_Population=("Population", "sum"))
    .reset_index()
)

# Compute crude rate
agg_age["Crude_Rate_per_100k"] = (
    agg_age["Total_Cases"] / agg_age["Total_Population"] * 100_000
).round(4)
agg_age["Crude_Rate_per_100k"] = agg_age["Crude_Rate_per_100k"].where(
    agg_age["Total_Population"] > 0, other=None
)

# Sort by period, sex, age group
agg_age["Period"] = pd.Categorical(agg_age["Period"], categories=period_order, ordered=True)
agg_age = agg_age.sort_values(["Period", "Sex", "Age_Group_Sort"]).reset_index(drop=True)

# Save (raw 18-age-group view)
agg_age.to_csv(OUT_AGE, index=False)

print("Age-Group Specific All-Sites Summary - Pune Registry\n")
print(agg_age.to_string(index=False))
print(f"\nSaved: {OUT_AGE}  ({len(agg_age)} rows)")

## Grouping Ages into Broader 15-Year Bands

This cell simplifies the 18 detailed age groups into 6 broader, easier-to-interpret age bands, and adds a population share percentage for each.

- It defines a mapping that collapses the original 18 age groups into 6 wider bands (0-14, 15-29, 30-44, 45-59, 60-74, and 75+), each spanning roughly 15 years.
- It applies this mapping to the age-group data from the previous step, tagging each row with its new broader age band and a sort order.
- It re-aggregates the case counts and population totals within each new band, combining the finer age groups that now fall under the same broader category.
- It recalculates the crude cancer rate per 100,000 people for each band, again leaving the rate blank if there's no recorded population to avoid a misleading calculation.
- It also calculates what percentage of the total population (within each period and sex) falls into each age band.
- Finally, it sorts the table by period, sex, and age band order, saves it as a CSV file, and prints the full result along with a row count for review.

In [ ]:
# Bucket 18 age groups into 6 fifteen-year bands
band_map = {
    "0-4": "0-14", "5-9": "0-14", "10-14": "0-14",
    "15-19": "15-29", "20-24": "15-29", "25-29": "15-29",
    "30-34": "30-44", "35-39": "30-44", "40-44": "30-44",
    "45-49": "45-59", "50-54": "45-59", "55-59": "45-59",
    "60-64": "60-74", "65-69": "60-74", "70-74": "60-74",
    "75-79": "75-79"}

band_order = ["0-14", "15-29", "30-44", "45-59", "60-74", "75-79"]
band_sort = {b: i + 1 for i, b in enumerate(band_order)}

agg_age2 = agg_age.copy()
agg_age2["Period"] = agg_age2["Period"].astype(str)   # avoid categorical cartesian expansion
agg_age2["Age_Band"] = agg_age2["Age_Group"].map(band_map)
agg_age2["Age_Band_Sort"] = agg_age2["Age_Band"].map(band_sort)

# Aggregate cases and population within each new 15-year band
agg_band = (
    agg_age2
    .groupby(["Period", "Sex", "Age_Band", "Age_Band_Sort"], sort=False, observed=True)
    .agg(Total_Cases=("Total_Cases", "sum"), Total_Population=("Total_Population", "sum"))
    .reset_index()
)

agg_band["Crude_Rate_per_100k"] = (
    agg_band["Total_Cases"] / agg_band["Total_Population"] * 100_000
).round(4)
agg_band["Crude_Rate_per_100k"] = agg_band["Crude_Rate_per_100k"].where(
    agg_band["Total_Population"] > 0, other=None
)

# Percent of each Period x Sex total population in each band (sums to 100)
period_sex_totals = agg_band.groupby(["Period", "Sex"], observed=True)["Total_Population"].transform("sum")
agg_band["Population_Pct"] = (agg_band["Total_Population"] / period_sex_totals * 100).round(4)

agg_band["Period"] = pd.Categorical(agg_band["Period"], categories=period_order, ordered=True)
agg_band = agg_band.sort_values(["Period", "Sex", "Age_Band_Sort"]).reset_index(drop=True)

# Save (bucketed 6-age-band view with percent shares)
agg_band.to_csv(OUT_AGEBAND15, index=False)

print("\n15-Year Age-Band Summary (% of Population) - Pune Registry\n")
print(agg_band.to_string(index=False))
print(f"\nSaved: {OUT_AGEBAND15}  ({len(agg_band)} rows)")

## Checking Which Cancer Labels Appear in Each Volume

This cell compares the cancer type names across all five volumes to see which labels show up consistently and which only appear in some.

- It collects the set of unique cancer labels used in each volume separately, then combines them into one list of every distinct label seen across all five volumes.
- It prints a quick count of the total unique labels overall, plus how many labels each individual volume contains, giving a sense of how consistent the naming has been over time.
- For every label in the main list, it builds a row marking "YES" in the column for each volume where that label appears, and leaves the cell blank for volumes where it's missing.
- This table is organized into a dataframe and saved as a CSV file.
- Finally, this cell prints a short preview of the first 10 rows immidiate verification.

In [ ]:
# ── Cancer label presence across volumes → CSV ────────────────────────────
vol_order = ["VIII", "IX", "X", "XI", "XII"]

# Build per-volume label sets
vol_labels = {vol: set(CANCER_DICTS[vol].values()) for vol in vol_order}

# All unique labels across all volumes (sorted)
all_labels = sorted({lbl for labels in vol_labels.values() for lbl in labels})

print(f"Total unique raw cancer labels across all volumes: {len(all_labels)}")
for vol in vol_order:
    print(f"  CI5-{vol}: {len(vol_labels[vol])} labels")

# ── Build rows: label present (YES) or absent (NO) per volume ────────────────
rows = []
for lbl in all_labels:
    rows.append({
        "Cancer_Label": lbl,
        **{f"CI5-{vol}": "YES" if lbl in vol_labels[vol] else ""
           for vol in vol_order}
    })

df_labels = pd.DataFrame(rows)

# OUT_LABELS defined in Inputs
df_labels.to_csv(OUT_LABELS, index=False)

print(f"\nSaved: cancer_label_presence_by_volume.csv  ({len(df_labels)} rows)")
print(f"\n  Preview (first 10 rows):")
print(df_labels.head(10).to_string(index=False))

## Finding Cancer Labels Common to All Volumes

This cell filters the earlier table of labeling consistency down to only the cancer types that were tracked consistently across every single volume, from VIII through XII.

- It reloads the presence/absence table created earlier (which marks "YES" wherever a cancer label appears in a given volume).
- It checks, label by label, whether every volume column shows "YES" — meaning that specific cancer type was recorded in all five time periods without exception.
- It keeps only those labels that pass this all-volumes check, discarding any cancer type that was missing from even one volume, since inconsistent labels can't be reliably compared across the full time span.
- It saves this shortened list of consistently-tracked cancer labels to its own CSV file.
- It prints a quick comparison of how many labels existed originally versus how many survived the filter.

In [ ]:
df = pd.read_csv(OUT_LABELS)

vol_cols = [c for c in df.columns if c != "Cancer_Label"]

mask_all = df[vol_cols].apply(lambda col: col.str.strip() == "YES").all(axis=1)
df_common = df[mask_all].reset_index(drop=True)

df_common.to_csv(OUT_COMMON, index=False)

print(f"Total labels in original file: {len(df)}")
print(f"Present in all 5 volumes (VIII-XII): {len(df_common)}")
print(f"Saved: {OUT_COMMON}")
df_common

## Building a main Table of Common Cancer Sites

This cell rebuilds a single main dataset, but restricts it to only the cancer types that were consistently tracked across all volumes, using the filtered list from the previous step.

- It loads the list of common cancer labels (the ones present in every volume) that was saved earlier.
- It goes through each volume's individual CSV file and keeps only the rows whose cancer label appears on that common list, discarding anything that wasn't tracked consistently over time; it prints how many rows were kept versus the original total for each volume.
- It stacks the filtered data from all five volumes into one combined table.
- It sorts this main table logically — first by cancer type, then by sex, then chronologically by time period, and finally by age group.
- It prints a summary showing the total number of rows, how many distinct common cancer types made it into the final table, and how many time periods are represented.
- It saves this main table as a CSV file and displays the first 20 rows.

In [ ]:
common_labels = pd.read_csv(OUT_COMMON)["Cancer_Label"].str.strip().tolist()

period_files = {
    "VIII": OUT_IACR_VIII, "IX": OUT_IACR_IX, "X": OUT_IACR_X,
    "XI": OUT_IACR_XI, "XII": OUT_IACR_XII,
}

dfs = []
for vol, fname in period_files.items():
    d = pd.read_csv(fname)
    d["Cancer_Label"] = d["Cancer_Label"].str.strip()
    d_filtered = d[d["Cancer_Label"].isin(common_labels)].copy()
    dfs.append(d_filtered)
    print(f"{vol}: {len(d_filtered)} rows kept (of {len(d)} total)")

df_MAIN = pd.concat(dfs, ignore_index=True)

# Exclude 80-84 and 85+ age groups (unreliable population data, skews crude rate)
n_before = len(df_MAIN)
df_MAIN = df_MAIN[~df_MAIN["Age_Group"].isin(["80-84", "85+"])].copy()
print(f"Excluded 80-84 and 85+ age groups: {n_before - len(df_MAIN)} rows removed")

df_MAIN = df_MAIN.sort_values(
    ["Cancer_Label", "Sex", "Year_Midpoint", "Age_Group_Sort"]
).reset_index(drop=True)

n_labels = df_MAIN["Cancer_Label"].nunique()
n_periods = df_MAIN["Period"].nunique()
print(f"\nFinal table: {len(df_MAIN)} rows, {n_labels} common cancer sites, {n_periods} periods")

df_MAIN.to_csv(OUT_TISSUE_MAIN, index=False)
print(f"Saved: {OUT_TISSUE_MAIN}")

df_MAIN.head(18)

## Identifying Top 10 Cancer Sites by Sex

This cell finds the 10 most common cancer types separately for males and females, since the leading cancers often differ significantly between sexes, then combines both lists into one trend dataset.

- It removes the broad "All sites" categories, then ranks cancer types by total case count separately within each sex, producing two independent top-10 lists (since a cancer type could rank high for one sex and not appear at all for the other, or vice versa).
- It prints both top-10 rankings for a quick sanity check before continuing.
- It filters the data down to just the male-specific top-10 sites for males and the female-specific top-10 sites for females, then stacks both filtered sets into one combined table.
- For each cancer type, sex, and time period, it sums total cases and population, then calculates the crude rate per 100,000 people.
- It attaches each cancer type's sex-specific rank (1st through 10th) back onto the trend table, so the final CSV clearly shows which sex's top-10 list each row belongs to and where it ranked.
- It sorts the table by sex, rank, and period, then saves the combined result as one CSV file, ready for separate or side-by-side visualization of male versus female leading cancers.

In [ ]:
df = pd.read_csv(OUT_TISSUE_MAIN)

exclude_labels = ["All sites", "All sites but skin"]
df_sites = df[~df["Cancer_Label"].isin(exclude_labels)].copy()

top10_by_sex = {}
for sex in ["Male", "Female"]:
    sex_totals = (df_sites[df_sites["Sex"] == sex]
                  .groupby("Cancer_Label")["Cases"].sum()
                  .sort_values(ascending=False)
                  .reset_index()
                  .rename(columns={"Cases": "Total_Cases_AllPeriods"}))
    sex_totals["Sex"] = sex
    sex_totals["Rank"] = range(1, len(sex_totals) + 1)
    top10_by_sex[sex] = sex_totals.head(10)

    print(f"Top 10 cancer sites by total cases ({sex}, 1993-2017):")
    print(top10_by_sex[sex][["Rank", "Cancer_Label", "Total_Cases_AllPeriods"]].to_string(index=False))
    print()

top10_labels_male   = top10_by_sex["Male"]["Cancer_Label"].tolist()
top10_labels_female = top10_by_sex["Female"]["Cancer_Label"].tolist()

df_top10_male   = df_sites[(df_sites["Sex"] == "Male")   & (df_sites["Cancer_Label"].isin(top10_labels_male))].copy()
df_top10_female = df_sites[(df_sites["Sex"] == "Female") & (df_sites["Cancer_Label"].isin(top10_labels_female))].copy()
df_top10 = pd.concat([df_top10_male, df_top10_female], ignore_index=True)

trend = (df_top10.groupby(["Cancer_Label", "Period", "Year_Midpoint", "Sex"])
         .agg(Total_Cases=("Cases", "sum"), Total_Population=("Population", "sum"))
         .reset_index())
trend["Crude_Rate_per_100k"] = trend["Total_Cases"] / trend["Total_Population"] * 100000

rank_lookup = pd.concat([top10_by_sex["Male"], top10_by_sex["Female"]])[["Cancer_Label", "Sex", "Rank"]]
trend = trend.merge(rank_lookup, on=["Cancer_Label", "Sex"], how="left")

trend = trend.sort_values(["Sex", "Rank", "Period"]).reset_index(drop=True)

trend.to_csv(OUT_TOP10, index=False)
print(f"Saved: {OUT_TOP10}")
trend.head(20)

## Building the Age-Distribution Data Table by Sex

This cell prepares the detailed age-specific rate data separately for the top 10 cancer types in males and the top 10 in females, saving both into one combined CSV.

- It reloads the master table, removes the unreliable "85+" age group, and excludes the broad "All sites" categories to focus on individual cancer types.
- It ranks cancer types by total case count separately within each sex, producing two independent top-10 lists, since the leading cancers can differ substantially between males and females.
- It filters the data down to just each sex's own top-10 sites, then combines both filtered sets into one dataset.
- It calculates the cancer rate per 100,000 people for every combination of cancer type, sex, time period, and individual age group (17 groups from "0-4" to "80-84").
- It orders the age groups correctly and sorts the table by cancer type, period, sex, and age, then saves this combined, sex-specific dataset as one CSV file.

In [ ]:
df = pd.read_csv(OUT_TISSUE_MAIN)

df = df[df["Age_Group"] != "85+"].copy()

exclude_labels = ["All sites", "All sites but skin"]
df_sites = df[~df["Cancer_Label"].isin(exclude_labels)].copy()

top10_by_sex = {}
for sex in ["Male", "Female"]:
    top10_by_sex[sex] = (df_sites[df_sites["Sex"] == sex]
                          .groupby("Cancer_Label")["Cases"].sum()
                          .sort_values(ascending=False).head(10).index.tolist())

df_top10_male   = df_sites[(df_sites["Sex"] == "Male")   & (df_sites["Cancer_Label"].isin(top10_by_sex["Male"]))].copy()
df_top10_female = df_sites[(df_sites["Sex"] == "Female") & (df_sites["Cancer_Label"].isin(top10_by_sex["Female"]))].copy()
df_top10 = pd.concat([df_top10_male, df_top10_female], ignore_index=True)

age_order = ["0-4","5-9","10-14","15-19","20-24","25-29","30-34","35-39","40-44",
             "45-49","50-54","55-59","60-64","65-69","70-74","75-79"]
period_order = ["1993-1997", "1998-2002", "2003-2007", "2008-2011", "2013-2017"]
periods_lbl = [p.replace("-", "\u2013") for p in period_order]

age_dist = (df_top10.groupby(["Cancer_Label", "Sex", "Period", "Age_Group", "Age_Group_Sort"])
            .agg(Total_Cases=("Cases", "sum"), Total_Population=("Population", "sum"))
            .reset_index())
age_dist["Rate_per_100k"] = age_dist["Total_Cases"] / age_dist["Total_Population"] * 100000
age_dist["Age_Group"] = pd.Categorical(age_dist["Age_Group"], categories=age_order, ordered=True)
age_dist = age_dist.dropna(subset=["Age_Group"]).sort_values(["Cancer_Label", "Period", "Sex", "Age_Group"])

age_dist.to_csv(OUT_AGE_DIST, index=False)
print(f"Saved: {OUT_AGE_DIST}")
trend.head(20)

## Setting a Consistent Look for All Charts

This cell it sets up shared styling rules so that every chart produced later in the script looks clean, consistent.

- It defines a handful of sizing values (font size, title size, line thickness, tick length and spacing) that will be reused across every figure.
- It updates Matplotlib's global settings to use a clean sans-serif font, a plain white background, and simplified chart borders (removing the top and right lines of each plot box, a common style choice in scientific publications).
- It sets tick marks, axis lines, and text to a consistent black color and thickness, and disables background grid lines for a cleaner, less cluttered look.
- It defines two specific colors — a blue for male data and a red/orange for female data — that will be used consistently in every chart that compares the sexes, making the figures easier to read at a glance.
- It also prepares a standard source citation line (crediting the IARC Cancer Incidence in Five Continents database) that can be added to the bottom of every chart for proper attribution.
- Finally, it prints a short confirmation that the styling has been applied successfully.

In [ ]:
# ── Publication style constants (shared across all figures) ─────────────
FONT_PT     = 11
FONT_AXIS   = 11
FONT_TITLE  = 13
LW_SPINE    = 0.8
TICK_LEN    = 3.0
TICK_PAD    = 3.0

plt.rcParams.update({
    'font.family':       'sans-serif',
    'font.sans-serif':   ['Arial', 'Helvetica', 'DejaVu Sans'],
    'font.size':         FONT_PT,
    'axes.titlesize':    FONT_TITLE,
    'axes.labelsize':    FONT_AXIS,
    'xtick.labelsize':   8,
    'ytick.labelsize':   FONT_PT,
    'figure.facecolor':  'white',
    'axes.facecolor':    'white',
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'axes.edgecolor':    'black',
    'axes.linewidth':    LW_SPINE,
    'axes.grid':         False,
    'xtick.color':       'black',
    'ytick.color':       'black',
    'xtick.major.size':  TICK_LEN,
    'ytick.major.size':  TICK_LEN,
    'xtick.major.width': LW_SPINE,
    'ytick.major.width': LW_SPINE,
    'xtick.major.pad':   TICK_PAD,
    'ytick.major.pad':   TICK_PAD,
    'xtick.direction':   'out',
    'ytick.direction':   'out',
    'legend.frameon':    False,
    'text.color':        'black',
    'pdf.fonttype':      42,
    'ps.fonttype':       42,
})

COLOR_MALE   = '#4393C3'
COLOR_FEMALE = '#D6604D'
SOURCE_NOTE  = ('Data: IARC Cancer Incidence in Five Continents (CI5) Vol. VIII–XII  ·  '
                'https://ci5.iarc.fr')
print('✓ Style constants applied')

## Charting Population Trends by Sex

This cell creates a line chart showing how the total population covered by the registry (the "population at risk") changed over time, split by male and female.

- It reloads the crude rate table and puts the five time periods in a chronological order rather than alphabetical order.
- It separates the data into two groups — male and female — each sorted by time period.
- It draws a line for males (in blue) and a line for females (in red/orange) across the five periods, with round markers at each data point, and labels each point directly on the chart showing the population in millions.
- It adds a light shaded area between the two lines to visually highlight the gap between male and female population sizes at each point in time.
- It formats the axes to make the y-axis shows population in millions (e.g. "1.5M") and the x-axis shows the year ranges, then adds a bold title, a legend distinguishing male from female, and a small source citation crediting the original IARC data.
- Finally, it saves the finished chart as a high-resolution PNG image file and displays it directly in the notebook.

In [ ]:
df_pop_fig = pd.read_csv(OUT_CRUDE_RATE)

period_order = ["1993-1997", "1998-2002", "2003-2007", "2008-2011", "2013-2017"]
df_pop_fig["Period"] = pd.Categorical(
    df_pop_fig["Period"], categories=period_order, ordered=True
)
df_pop_fig = df_pop_fig.sort_values(["Period", "Sex"]).reset_index(drop=True)

male_pop   = df_pop_fig[df_pop_fig["Sex"] == "Male"].sort_values("Period")
female_pop = df_pop_fig[df_pop_fig["Sex"] == "Female"].sort_values("Period")

x_vals      = np.arange(len(period_order))
periods_lbl = [p.replace("-", "\u2013") for p in period_order]

fig, ax = plt.subplots(figsize=(11, 6), facecolor="white")

# Lines
ax.plot(x_vals, male_pop["Total_Population"].values,
        color=COLOR_MALE, linewidth=2.2, marker="o", markersize=7,
        markerfacecolor=COLOR_MALE, markeredgecolor="white", markeredgewidth=1.5,
        zorder=3, label="Male")

ax.plot(x_vals, female_pop["Total_Population"].values,
        color=COLOR_FEMALE, linewidth=2.2, marker="o", markersize=7,
        markerfacecolor=COLOR_FEMALE, markeredgecolor="white", markeredgewidth=1.5,
        zorder=3, label="Female")

# Value labels
y_max = max(male_pop["Total_Population"].max(), female_pop["Total_Population"].max())
for xi, val in zip(x_vals, male_pop["Total_Population"].values):
    ax.text(xi, val + y_max * 0.015, f"{val/1_000_000:.2f}M",
            ha="center", va="bottom", fontsize=9, color=COLOR_MALE, fontweight="bold")
for xi, val in zip(x_vals, female_pop["Total_Population"].values):
    ax.text(xi, val - y_max * 0.035, f"{val/1_000_000:.2f}M",
            ha="center", va="top", fontsize=9, color=COLOR_FEMALE, fontweight="bold")

# Shaded gap between male and female
ax.fill_between(x_vals,
                male_pop["Total_Population"].values,
                female_pop["Total_Population"].values,
                alpha=0.08, color="gray", zorder=1)

# Axes
ax.set_xticks(x_vals)
ax.set_xticklabels(periods_lbl, fontsize=FONT_PT, color="black")
ax.set_xlabel("Registration Period", fontsize=FONT_AXIS, fontweight="bold",
              labelpad=10, color="black")
ax.yaxis.set_major_formatter(
    mticker.FuncFormatter(lambda v, _: f"{v/1_000_000:.1f}M"))
ax.set_ylabel("Population at Risk (Millions)",
              fontsize=FONT_AXIS, fontweight="bold", labelpad=8, color="black")
ax.set_xlim(-0.4, len(period_order) - 0.6)
ax.set_ylim(0, y_max * 1.18)
ax.tick_params(axis="both", colors="black", labelcolor="black",
               length=TICK_LEN, width=LW_SPINE, pad=TICK_PAD)
for sp in ["left", "bottom"]:
    ax.spines[sp].set_color("black")
    ax.spines[sp].set_linewidth(LW_SPINE)

# Title, legend & source note
ax.set_title("Population at Risk by Sex",
             loc="center", pad=10, fontsize=14, fontweight="bold", color="black")
ax.legend(loc="upper left", bbox_to_anchor=(0.01, 0.97), frameon=False,
          fontsize=FONT_PT, handlelength=1.5, handletextpad=0.5)
fig.text(0.01, 0.01, SOURCE_NOTE, ha="left", va="bottom",
         fontsize=8, color="gray", style="italic", transform=fig.transFigure)

plt.tight_layout(rect=[0, 0.04, 1, 1])
fig.savefig(OUT_FIG_POP, dpi=300, bbox_inches="tight", facecolor="white")
print(f"✓ Saved: {OUT_FIG_POP}")
plt.show()

## Calculating Population Growth from First to Last Period

This cell measures how much the population at risk grew between the earliest period (1993-1997) and the most recent period (2013-2017), separately for males and females.

- It pulls the total population figures for the first and last registration periods from the crude rate table, for both sexes.
- It calculates the standard percent change between these two points, showing growth as a percentage (e.g., "grew by 114%").
- Since a percent change this large can be hard to picture intuitively, it also calculates a "times change" ratio (e.g., "2.14x"), which more directly conveys that the population roughly doubled over this 20-year span.
- It packages both metrics into a small summary table — one row per sex — and prints the result, making it easy to choose whichever framing reads most clearly when annotating the population trend chart.

In [ ]:
df_pop_change = pd.read_csv(OUT_CRUDE_RATE)

first_period = "1993-1997"
last_period  = "2013-2017"

pop_change_rows = []
for sex in ["Male", "Female"]:
    pop_first = df_pop_change[(df_pop_change["Period"] == first_period) &
                               (df_pop_change["Sex"] == sex)]["Total_Population"].values[0]
    pop_last  = df_pop_change[(df_pop_change["Period"] == last_period) &
                               (df_pop_change["Sex"] == sex)]["Total_Population"].values[0]

    pct_change = round((pop_last - pop_first) / pop_first * 100, 2)
    times_change = round(pop_last / pop_first, 2)

    pop_change_rows.append({
        "Sex": sex,
        "Population_1993_1997": int(pop_first),
        "Population_2013_2017": int(pop_last),
        "Percent_Change": pct_change,
        "Times_Change": times_change,
        "Times_Change_Label": f"{times_change}x",
    })

df_pop_change_summary = pd.DataFrame(pop_change_rows)
print(df_pop_change_summary.to_string(index=False))

## Charting Crude Cancer Rates by Sex

This cell creates a grouped bar chart comparing overall cancer case rates (per 100,000 people) between males and females across each of the five registration periods.

- It reloads the crude rate table and reshapes it for male and female rates to sit side by side for each time period, then puts the periods in proper chronological order.
- It sets up the bar positions such that each time period gets two bars placed close together — one for males, one for females — with a small gap between the pairs for visual clarity.
- It draws the bars in the standard blue (male) and red/orange (female) colors, and prints the exact rate value directly above each bar.
- It formats the axes with the registration periods along the bottom and the cancer case rate per 100,000 along the side.
- It adds a title, a legend distinguishing male from female bars, and the standard source citation at the bottom of the figure.
- Finally, it saves the chart as a high-resolution PNG file and displays it in the notebook for review.

In [ ]:
# Data: load pre-computed crude rates
df_m = pd.read_csv(OUT_CRUDE_RATE)

period_order = ["1993-1997", "1998-2002", "2003-2007", "2008-2011", "2013-2017"]
df_m["Period"] = pd.Categorical(df_m["Period"], categories=period_order, ordered=True)

pivot       = df_m.pivot(index="Period", columns="Sex", values="Crude_Rate_per_100k")
pivot       = pivot.reindex(period_order)
periods     = pivot.index.tolist()
periods_lbl = [p.replace("-", "\u2013") for p in periods]
male_vals   = pivot["Male"].values
female_vals = pivot["Female"].values

# Layout
x        = np.arange(len(periods))
bar_w    = 0.36
gap      = 0.04
x_male   = x - bar_w / 2 - gap / 2
x_female = x + bar_w / 2 + gap / 2

fig, ax = plt.subplots(figsize=(11, 6), facecolor='white')

bars_m = ax.bar(x_male,   male_vals,   width=bar_w, color=COLOR_MALE,
                alpha=0.88, edgecolor='none', zorder=3, label='Male')
bars_f = ax.bar(x_female, female_vals, width=bar_w, color=COLOR_FEMALE,
                alpha=0.88, edgecolor='none', zorder=3, label='Female')

# Value labels
for bar, val in zip(bars_m, male_vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 3,
            f'{val:.1f}', ha='center', va='bottom',
          fontsize=10, color='black', fontweight='bold')
for bar, val in zip(bars_f, female_vals):
      ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 3,
            f'{val:.1f}', ha='center', va='bottom',
           fontsize=10, color='black', fontweight='bold')

# Axes
ax.set_xticks(x)
ax.set_xticklabels(periods_lbl, fontsize=FONT_PT, color='black')
ax.set_xlabel('Registration Period', fontsize=FONT_AXIS, fontweight='bold',
              labelpad=10, color='black')
ax.yaxis.set_major_formatter(
    mticker.FuncFormatter(lambda v, _: f'{v:.0f}'))
ax.set_ylabel('Crude Rate per 100,000', fontsize=FONT_AXIS,
              fontweight='bold', labelpad=8, color='black')
ax.set_xlim(-0.65, len(periods) - 0.35)
ax.set_ylim(0, max(male_vals.max(), female_vals.max()) * 1.22)
ax.tick_params(axis='both', colors='black', labelcolor='black',
               length=TICK_LEN, width=LW_SPINE, pad=TICK_PAD)
for sp in ['left', 'bottom']:
    ax.spines[sp].set_color('black')
    ax.spines[sp].set_linewidth(LW_SPINE)

# Title, legend & source note
ax.set_title('Crude Cancer Rate per 100,000 by Sex',
             loc='center', pad=10, fontsize=14, fontweight='bold', color='black')
ax.legend(loc='upper left', bbox_to_anchor=(0.01, 0.97), frameon=False,
          fontsize=FONT_PT, handlelength=1.0, handletextpad=0.5)
fig.text(0.01, 0.01, SOURCE_NOTE, ha='left', va='bottom',
         fontsize=8, color='gray', style='italic', transform=fig.transFigure)

plt.tight_layout(rect=[0, 0.04, 1, 1])
fig.savefig(OUT_FIG_SEX, dpi=300, bbox_inches='tight', facecolor='white')
print(f'✓  Saved: {OUT_FIG_SEX}')
plt.show()

## Charting Age Distribution as Population Share

This cell creates a pair of 100%-stacked bar charts — one for females, one for males — showing how the population's age makeup shifted across the five registration periods.

- It assigns a distinct color to each of the six age bands (from "0-14" up to "75+").
- It builds two side-by-side charts, one for each sex, where each bar represents 100% of the population in that time period, divided into colored segments showing what share belonged to each age band.
- For each age band, it stacks its percentage on top of the previous bands within the same bar, and labels any segment large enough (over 4%) with its exact percentage.
- It formats both charts with matching axes, labels the y-axis as "Percent of Population," and titles each chart "Female" or "Male" accordingly.
- It adds one shared legend above both charts explaining what each color/age band represents, plus an overall title and a source citation crediting the Pune registry data.
- Finally, it saves the combined two-panel figure as a high-resolution PNG file and displays it in the notebook.

In [ ]:
colors = {
    "0-14": "#2E5EAA", "15-29": "#4DA9D6", "30-44": "#5EC49A",
    "45-59": "#E8B84B", "60-74": "#E17B3B", "75-79": "#B8412F",
}

FONT_PT, FONT_AXIS, TICK_LEN, TICK_PAD, LW_SPINE = 10, 12, 4, 6, 1.0
SOURCE_NOTE = "Source: IACR Pune Registry, Volumes VIII-XII (1993-2017)"

periods_lbl = [p.replace("-", "\u2013") for p in period_order]
x = np.arange(len(period_order))
bar_w = 0.36
gap = 0.04

fig, axes = plt.subplots(1, 2, figsize=(13, 6), facecolor='white', sharey=True)

for ax, sex, title in zip(axes, ["Female", "Male"], ["Female", "Male"]):
    bottoms = np.zeros(len(period_order))
    for band in band_order:
        vals = []
        for p in period_order:
            row = agg_band[(agg_band["Period"] == p) & (agg_band["Sex"] == sex) & (agg_band["Age_Band"] == band)]
            vals.append(row["Population_Pct"].values[0] if len(row) else 0)
        vals = np.array(vals)
        ax.bar(x, vals, width=bar_w * 1.6, bottom=bottoms, color=colors[band],
               edgecolor='white', linewidth=0.5, zorder=3, label=band)
        for xi, v, b in zip(x, vals, bottoms):
            if v > 4:
                ax.text(xi, b + v / 2, f'{v:.0f}%', ha='center', va='center',
                        fontsize=8, color='white', fontweight='bold')
        bottoms += vals
    ax.set_xticks(x)
    ax.set_xticklabels(periods_lbl, fontsize=FONT_PT, color='black', rotation=0)
    ax.set_xlabel('Registration Period', fontsize=FONT_AXIS, fontweight='bold', labelpad=10, color='black')
    ax.set_title(title, fontsize=13, fontweight='bold', color='black')
    ax.set_ylim(0, 100)
    ax.tick_params(axis='both', colors='black', labelcolor='black', length=TICK_LEN, width=LW_SPINE, pad=TICK_PAD)
    for sp in ['left', 'bottom']:
        ax.spines[sp].set_color('black')
        ax.spines[sp].set_linewidth(LW_SPINE)
    for sp in ['top', 'right']:
        ax.spines[sp].set_visible(False)

axes[0].set_ylabel('Percent of Population', fontsize=FONT_AXIS, fontweight='bold', labelpad=8, color='black')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:.0f}%'))

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 1.04),
           ncol=6, frameon=False, fontsize=FONT_PT, title="Age Groups")

fig.suptitle('Pune Share of Population by Sex', fontsize=14, fontweight='bold', y=1.12)
fig.text(0.01, 0.01, SOURCE_NOTE, ha='left', va='bottom', fontsize=8, color='gray', style='italic', transform=fig.transFigure)

plt.tight_layout(rect=[0, 0.04, 1, 0.95])

fig.savefig(OUT_FIG_AGEBAND, dpi=300, bbox_inches='tight', facecolor='white')
print(f'✓  Saved: {OUT_FIG_AGEBAND}')
plt.show()

## Charting Cancer Rates by Age Band and Sex

This cell builds a detailed clustered bar chart showing the actual cancer rate (not just population share) for each age band, split by sex, across all five time periods.

- For each registration period, it creates a small cluster of bars — one for every age band — and within each age band, it places two bars side by side representing female and male rates.
- It reuses the same age-band colors as the previous chart for consistency, but distinguishes males from females using a hatched (striped) pattern for males and a solid fill for females.
- For every bar, it prints the actual crude rate value rotated vertically above the bar.
- It labels the x-axis to explain the pairing convention (female bar on the left, male bar on the right within each age-band group) and labels the y-axis as "Crude Rate per 100k".
- It builds two separate legends off to the side of the chart — one explaining which color corresponds to which age band, and another explaining that solid bars mean female while hatched bars mean male.
- It adds a descriptive title, a source citation, and saves the finished chart as a high-resolution PNG file before displaying it in the notebook.

In [ ]:
fig2, ax = plt.subplots(figsize=(17, 6.5), facecolor='white')

n_bands = len(band_order)
sexes = ["Female", "Male"]

cluster_w = 0.8
band_slot_w = cluster_w / n_bands
bar_w2 = band_slot_w / 2

sex_offset = {"Female": -bar_w2/2, "Male": bar_w2/2}

for i, band in enumerate(band_order):
    band_center = (i - (n_bands - 1) / 2) * band_slot_w
    for si, sex in enumerate(sexes):
        vals = []
        for p in period_order:
            row = agg_band[(agg_band["Period"] == p) & (agg_band["Sex"] == sex) & (agg_band["Age_Band"] == band)]
            v = row["Crude_Rate_per_100k"].values[0] if len(row) else np.nan
            vals.append(v if pd.notna(v) else 0)
        offset = band_center + sex_offset[sex]
        ax.bar(x + offset, vals, width=bar_w2 * 0.92, color=colors[band],
               edgecolor='white', linewidth=0.4, zorder=3,
               label=band if si == 0 else None,
               hatch='//' if sex == "Male" else None)
        for xi, v in zip(x + offset, vals):
            if v > 0:
                ax.text(xi, v + 1, f'{v:.0f}', ha='center', va='bottom',
                        fontsize=8.5, color='black', rotation=90, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(periods_lbl, fontsize=FONT_PT, color='black')
ax.set_xlabel('Registration Period',
              fontsize=FONT_AXIS, fontweight='bold', labelpad=10, color='black')
ax.set_ylabel('Crude Rate per 100k', fontsize=FONT_AXIS, fontweight='bold', labelpad=8, color='black')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:.0f}'))
ax.tick_params(axis='both', colors='black', labelcolor='black', length=TICK_LEN, width=LW_SPINE, pad=TICK_PAD)
for sp in ['left', 'bottom']:
    ax.spines[sp].set_color('black')
    ax.spines[sp].set_linewidth(LW_SPINE)
for sp in ['top', 'right']:
    ax.spines[sp].set_visible(False)

age_handles, age_labels = ax.get_legend_handles_labels()
sex_handles = [mpatches.Patch(facecolor='white', edgecolor='black', label='Female (solid)'),
               mpatches.Patch(facecolor='white', edgecolor='black', hatch='//', label='Male (hatched)')]
leg1 = ax.legend(handles=age_handles, labels=age_labels, title="Age Band",
                 loc='upper left', bbox_to_anchor=(1.01, 1.0), frameon=False, fontsize=9)
ax.add_artist(leg1)
ax.legend(handles=sex_handles, loc='upper left', bbox_to_anchor=(1.01, 0.35), frameon=False, fontsize=9)

ax.set_title('Pune Crude Cancer Rate per 100k',
             fontsize=14, fontweight='bold', color='black', pad=12)

fig2.text(0.01, 0.01, SOURCE_NOTE, ha='left', va='bottom', fontsize=8, color='gray',
          style='italic', transform=fig2.transFigure)

plt.tight_layout(rect=[0, 0.04, 0.86, 1])

fig2.savefig(OUT_FIG_AGEBAND_RATE, dpi=300, bbox_inches='tight', facecolor='white')
print(f'Saved: {OUT_FIG_AGEBAND_RATE}')
plt.show()

## Charting Top 10 Cancer Sites by Sex

This cell produces two separate grids of small charts — one for females' top 10 cancer sites, one for males' — since the leading cancers differ by sex and each list is now independently ranked.

- It reloads the sex-specific top-10 trend table and separates it into a female subset and a male subset.
- For each sex, it ranks that sex's cancer types by total case count, then builds a 2-row-by-5-column grid of small bar charts, one panel per cancer type, showing how its crude rate changed across the five registration periods.
- Each panel is titled with the cancer type's name, and bars are colored consistently (red/orange for the female figure, blue for the male figure) to match the rest of the notebook's styling.
- It adds an overall title identifying which sex's top 10 are being shown, a shared y-axis label, and a source citation on each figure.
- Both figures are saved as separate high-resolution PNG files and displayed one after another in the notebook.

In [ ]:
trend = pd.read_csv(OUT_TOP10)

period_order = ["1993-1997", "1998-2002", "2003-2007", "2008-2011", "2013-2017"]
periods_lbl = [p.replace("-", "\u2013") for p in period_order]
x = np.arange(len(period_order))



def plot_top10_by_sex(sex, color, out_path, fig_label):
    sub_sex = trend[trend["Sex"] == sex].copy()
    site_order = (sub_sex.sort_values("Rank")["Cancer_Label"].unique().tolist())

    fig, axes = plt.subplots(nrows=2, ncols=5, figsize=(22, 9), facecolor='white', sharex=True)
    axes_flat = axes.flatten()

    for ax, site in zip(axes_flat, site_order):
        sub = sub_sex[sub_sex["Cancer_Label"] == site]
        vals = sub.set_index("Period").reindex(period_order)["Crude_Rate_per_100k"].fillna(0).values

        ax.bar(x, vals, width=0.5, color=color, edgecolor='white', linewidth=0.4, zorder=3)

        ax.set_title(site, fontsize=10.5, fontweight='bold', color='black', pad=6)
        ax.set_xticks(x)
        ax.set_xticklabels(periods_lbl, fontsize=7, rotation=45, ha='right', color='black')
        ax.tick_params(axis='y', labelsize=8, colors='black')
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:.0f}'))
        for sp in ['top', 'right']:
            ax.spines[sp].set_visible(False)
        for sp in ['left', 'bottom']:
            ax.spines[sp].set_color('black')
            ax.spines[sp].set_linewidth(0.8)

    for j in range(len(site_order), len(axes_flat)):
        axes_flat[j].set_visible(False)

    fig.suptitle(f'Crude Rate per 100k for Top 10 Cancer Sites \u2014 {fig_label}',
                 fontsize=15, fontweight='bold', y=1.05)
    fig.text(0.01, 0.005, SOURCE_NOTE, ha='left', va='bottom', fontsize=8, color='gray',
             style='italic', transform=fig.transFigure)
    fig.text(0.005, 0.5, 'Crude Rate per 100,000', va='center', rotation='vertical',
             fontsize=12, fontweight='bold', color='black')

    plt.tight_layout(rect=[0.02, 0.02, 1, 0.95])
    fig.savefig(out_path, dpi=300, bbox_inches='tight', facecolor='white')
    print(f'Saved: {out_path}')
    plt.show()

plot_top10_by_sex("Female", COLOR_FEMALE, OUT_TOP10_FIG_FEMALE, "Female")
plot_top10_by_sex("Male", COLOR_MALE, OUT_TOP10_FIG_MALE, "Male")

## Charting Age-Specific Rate Curves by Sex

This cell takes the sex-specific age-distribution data and produces two separate multi-panel figures — one for females' top 10 cancer sites, one for males' — each showing how the rate changes across every age group and time period.

- For each sex, it ranks that sex's cancer types by total case count, then builds a 2-row-by-5-column grid of line charts, one panel per cancer type.
- Each panel shows five lines, one per time period, colored on a gradient scale, tracing the cancer rate across the full range of age groups for that sex only.
- It labels each panel with the cancer type's name and adds a single legend explaining which color corresponds to which time period.
- It adds an overall title identifying which sex's top 10 are being shown, a shared y-axis label, and a source note mentioning that the 85+ age group was excluded.
- Both figures are saved as separate high-resolution PNG files and displayed one after another in the notebook.

In [ ]:
period_colors = {p: cm.viridis(i / (len(period_order) - 1)) for i, p in enumerate(period_order)}
x = np.arange(len(age_order))

def plot_age_dist_by_sex(sex, out_path, fig_label):
    sub_sex = age_dist[age_dist["Sex"] == sex].copy()
    site_order = (sub_sex.groupby("Cancer_Label")["Total_Cases"].sum()
                  .sort_values(ascending=False).index.tolist())

    fig, axes = plt.subplots(nrows=2, ncols=5, figsize=(24, 10), facecolor='white', sharex=True)
    axes_flat = axes.flatten()

    for ax, site in zip(axes_flat, site_order):
        sub = sub_sex[sub_sex["Cancer_Label"] == site]
        for period in period_order:
            ssub = sub[sub["Period"] == period].set_index("Age_Group").reindex(age_order)
            vals = ssub["Rate_per_100k"].fillna(0).values
            ax.plot(x, vals, linestyle='-', marker='o', markersize=2.8,
                    linewidth=1.3, color=period_colors[period], alpha=0.9, zorder=3)

        ax.set_title(site, fontsize=10.5, fontweight='bold', color='black', pad=6)
        ax.set_xticks(x)
        ax.set_xticklabels(age_order, rotation=45, ha='right', fontsize=6, color='black')
        ax.tick_params(axis='y', labelsize=8, colors='black')
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:.0f}'))
        for sp in ['top', 'right']:
            ax.spines[sp].set_visible(False)
        for sp in ['left', 'bottom']:
            ax.spines[sp].set_color('black')
            ax.spines[sp].set_linewidth(0.8)

    for j in range(len(site_order), len(axes_flat)):
        axes_flat[j].set_visible(False)

    period_handles = [mlines.Line2D([], [], color=period_colors[p], linewidth=2.5, label=periods_lbl[i])
                       for i, p in enumerate(period_order)]
    fig.legend(handles=period_handles, loc='upper left', bbox_to_anchor=(0.85, 0.92),
               frameon=False, fontsize=9.5, title="Period")

    fig.suptitle(f'Age Distribution of Cancer Incidence Rate per 100k for Top 10 Cancer Sites, {fig_label}',
                 fontsize=15, fontweight='bold', y=1.0, x=0.42)
    fig.text(0.01, 0.005, SOURCE_NOTE + " 85+ age band excluded (no reliable population data).",
             ha='left', va='bottom', fontsize=8, color='gray', style='italic', transform=fig.transFigure)
    fig.text(0.005, 0.5, 'Rate per 100,000', va='center', rotation='vertical',
             fontsize=12, fontweight='bold', color='black')

    plt.tight_layout(rect=[0.02, 0.02, 0.84, 0.96])
    fig.savefig(out_path, dpi=300, bbox_inches='tight', facecolor='white')
    print(f'Saved: {out_path}')
    plt.show()

plot_age_dist_by_sex("Female", OUT_AGE_FIG_FEMALE, "Female")
plot_age_dist_by_sex("Male", OUT_AGE_FIG_MALE, "Male")

## Key Conclusions

- Both the male and female population at risk between 0-79 years of age in Pune grew by 2.15 and 2.13 times for males and females respectively between 1993 and 2017.
- All-sites crude cancer incidence rates per 100k among females remained consistently higher than male rates across registration periods from 1993 to 2017.
- All-sites crude cancer incidence rates per 100k increased with age in males and females within each registration period from 1993-2017.
- Collective analysis of top 10 crude cancer incidence rates per 100k after filtering for consistent cancer names across registration periods, shows that colon, mouth, oesophagus, rectum, stomach, and tongue were tissues commonly diagnosed with cancer between males and females, whereas, bladder, larynx, liver, and prostate in males and breast, cervix uteri, corpus uteri, ovary in females were diagnosed at a higher rate.
- Age-specific trends of crude cancer incidence rates per 100k for top 10 tissues between males and females show an overall increase in cancer incidence rates with age.